# 22 · Geographic distance features (experiment)

## 1. Introduction

**Question:** do straight-line distances to a few fixed places in Hyderabad (the city centre, the airport and the main
employment areas) help Ridge predict the asking price?

This is an **experiment only**. The features are **not** added to the final model automatically, no data file is
changed, no model or transformed data is saved, and the **test set is not used**.

### Why distances might help

Ridge already knows *where* a home is in two ways (notebook 18):

- the **one-hot locality columns** (localities with ≥ 10 listings in the fold's training part; the rest share one
  "rare" column);
- the **locality centre coordinates** `locality_centre_lat` / `locality_centre_long`, which enter Ridge **linearly**.

A linear term in latitude can only say "prices rise (or fall) steadily as you go north". Hyderabad prices do not behave
like that: they are highest around the western IT corridor (HITEC City, Gachibowli, Financial District) and fall away
from it in **every** direction. Distance to a place can express "closer is more expensive" in all directions with one
number, which a linear model cannot build from latitude and longitude. Distances are most likely to help the **rare**
localities, which have no one-hot column of their own.

### Which coordinates: listing pin or locality centre?

| | Listing pin (`latitude` / `longitude`) | Locality centre (`locality_centre_lat` / `_long`) |
|---|---|---|
| source | typed/placed by the advertiser | the website's locality table, one fixed point per locality |
| training coverage (notebook 17, section 6.4) | 10,266 of 16,219 have a pin (not (0,0)) | 15,016 matched + 85 recovered |
| quality | 4,351 pins are just a copy of the locality centre, 71 are outside Hyderabad, 264 are > 20 km from their locality; only **5,587 (34%)** add plausible detail | 20 listings in 9 localities had a centre outside the Hyderabad box; notebook 18 set those to missing |
| used in the baseline? | no (rejected in notebook 17) | yes |

**This notebook uses only the cleaned locality-centre coordinates** produced by `add_fixed_features()`. The listing
pins are not read at all. Section 2 re-counts the pin problems on the training set, so the choice is shown, not just
assumed.

### The four representations compared

The notebook-19 baseline **already contains** latitude/longitude (the locality centre). "Baseline + latitude/longitude"
would add the same two columns a second time. So we compare four versions of **the same** baseline that differ only in
the geographic columns:

| Model | Geographic columns | Answers |
|---|---|---|
| **A. Baseline Ridge** (exact notebook 19) | centre latitude + longitude | the reference |
| **B. No coordinates** | none (A minus latitude/longitude) | how much do latitude/longitude contribute at all? |
| **C. Distances instead of lat/long** | 6 distances | is the distance representation better than the lat/long one? |
| **D. Lat/long + distances** | latitude + longitude + 6 distances | does adding distances to the baseline help? |

Everything else (the other 19 features, preprocessing, `Ridge(alpha=1)`, log target, folds, metrics) is identical. No
locality target encoding (notebook 21) and no tuning.

A second, smaller comparison adds **one distance at a time** to the baseline, to see which distances carry signal and
which should be rejected.

### Decision rule, fixed before looking at the results

A geographic representation is recommended for the final model only if, compared with **A**:

1. mean CV **MAE is at least 1% lower** (≈ ₹0.7 lakh). For reference, notebook 21's 0.5% gain was judged too small to
   justify extra complexity;
2. **MAE and Median APE are both lower in at least 4 of the 5 folds**;
3. neither apartment-like types nor houses/villas get clearly worse (Median APE up by more than 0.5 points).

If these are not met, the recommendation is **not** to add the features.

## 2. Load data and lock the test set

The same loading steps as notebooks 19–21. MD5 fingerprints of the source files are taken now and checked at the end.

In [1]:
from pathlib import Path
import hashlib
import sys

import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent

CLEANED_CSV = PROJECT_ROOT / "data" / "processed" / "cleaned_listings.csv"
SPLIT_CSV = PROJECT_ROOT / "data" / "processed" / "train_test_split.csv"
LOCALITIES_CSV = PROJECT_ROOT / "data" / "extracted" / "localities.csv"
FEATURES_PY = PROJECT_ROOT / "src" / "features" / "build_features.py"

sys.path.insert(0, str(PROJECT_ROOT))
from src.features.build_features import HYDERABAD_LAT_RANGE, HYDERABAD_LONG_RANGE, INPUT_COLUMNS, add_fixed_features


def file_md5(path):
    # A fingerprint of the file's bytes. If the file changes, the fingerprint changes.
    return hashlib.md5(path.read_bytes()).hexdigest()


md5_at_start = {}
for path in [CLEANED_CSV, SPLIT_CSV, LOCALITIES_CSV, FEATURES_PY]:
    md5_at_start[path] = file_md5(path)
    print(f"{path.name:28} {md5_at_start[path]}")

cleaned_listings.csv         3a81a385a9aca28d5b70322e1e143c08
train_test_split.csv         10693e86d1d8a55e606399d718a8f556
localities.csv               8d9bb54dfc2fb49673e74b976ab73ace
build_features.py            ec9ba82a80e12d99798e512f510bfa84


In [2]:
listings = pd.read_csv(CLEANED_CSV, low_memory=False)
split_assignment = pd.read_csv(SPLIT_CSV)
locality_table = pd.read_csv(LOCALITIES_CSV, usecols=["locality_id", "locality_name", "locality_lat", "locality_long"])

listings = listings.merge(split_assignment[["property_id", "split"]], on="property_id", validate="one_to_one")

assert len(listings) == 20300
assert (listings["split"] == "train").sum() == 16219
assert (listings["split"] == "test").sum() == 4081
train_groups = set(listings.loc[listings["split"] == "train", "property_group_id"])
test_groups = set(listings.loc[listings["split"] == "test", "property_group_id"])
assert train_groups.isdisjoint(test_groups), "A property group is on both sides!"

train = listings[listings["split"] == "train"].copy()
test_property_ids = set(listings.loc[listings["split"] == "test", "property_id"])
test_row_count = len(test_property_ids)
del listings   # the test rows are gone from memory from here on

print("Training rows used in this experiment:", len(train))
print("Test rows locked (never used):", test_row_count)
print("property_group_id values on both sides:", len(train_groups & test_groups))

Training rows used in this experiment: 16219
Test rows locked (never used): 4081
property_group_id values on both sides: 0


## 3. Fixed landmarks and the Haversine distance

### The landmarks

Six fixed points, **written down in advance from public maps** (rounded to about 0.1–0.5 km). They are not computed from
this dataset, its prices or its listings.

| Landmark | Latitude | Longitude | What the point is |
|---|---|---|---|
| City centre | 17.3850 | 78.4867 | the standard coordinate for Hyderabad city (Abids–Koti area, about 2.5 km north of Charminar) |
| Airport (RGIA) | 17.2403 | 78.4294 | Rajiv Gandhi International Airport, Shamshabad |
| HITEC City | 17.4504 | 78.3808 | Cyber Towers, HITEC City |
| Gachibowli | 17.4401 | 78.3489 | Gachibowli |
| Financial District | 17.4170 | 78.3400 | Financial District, Nanakramguda |
| Secunderabad | 17.4337 | 78.5016 | Secunderabad railway station |

Three of these (HITEC City, Gachibowli, Financial District) lie within about 5 km of each other in the western IT
corridor, so their distances will be very similar. Section 4 measures how similar, and section 7 tests each one alone.

### The Haversine formula

The Earth is (almost) a sphere, so the straight-line distance between two points given in degrees is the length of
the **arc** along the sphere between them. The Haversine formula computes it:

1. convert all latitudes and longitudes from degrees to **radians** (the trigonometric functions need radians);
2. `a = sin²(Δlat / 2) + cos(lat₁) · cos(lat₂) · sin²(Δlong / 2)`
   (`Δlong` is multiplied by `cos(lat)` because lines of longitude get closer together away from the equator);
3. `central angle = 2 · arcsin(√a)`: the angle between the two points, seen from the Earth's centre;
4. `distance = Earth radius × central angle`, with the mean Earth radius **6,371 km**.

It is a "crow flies" distance: it ignores roads, traffic and travel time. Within a city the difference from a flat-map
formula (as used in notebook 17) is tiny. Haversine is used because it is the standard, exact formula for this and is
just as short.

In [3]:
EARTH_RADIUS_KM = 6371.0

# Fixed landmark coordinates (degrees), from public maps. Written down before any modelling; never derived from the data.
LANDMARKS = {
    "city_centre":        (17.3850, 78.4867),
    "airport":            (17.2403, 78.4294),
    "hitec_city":         (17.4504, 78.3808),
    "gachibowli":         (17.4401, 78.3489),
    "financial_district": (17.4170, 78.3400),
    "secunderabad":       (17.4337, 78.5016),
}
DISTANCE_FEATURES = ["distance_km_" + name for name in LANDMARKS]


def haversine_km(lat_1, long_1, lat_2, long_2):
    # Great-circle ("crow flies") distance in km. Works on single numbers and on whole pandas columns.
    lat_1, long_1 = np.radians(lat_1), np.radians(long_1)
    lat_2, long_2 = np.radians(lat_2), np.radians(long_2)
    a = np.sin((lat_2 - lat_1) / 2) ** 2 + np.cos(lat_1) * np.cos(lat_2) * np.sin((long_2 - long_1) / 2) ** 2
    central_angle = 2 * np.arcsin(np.sqrt(a))
    return EARTH_RADIUS_KM * central_angle


def add_distance_features(features):
    # One column per landmark: distance from the listing's LOCALITY CENTRE, in km.
    # Uses only the two centre columns, so it is a fixed rule per listing (nothing learned, no price).
    # A listing without a usable centre gets NaN; the pipeline imputes it inside each fold.
    distances = pd.DataFrame(index=features.index)
    for name, (landmark_lat, landmark_long) in LANDMARKS.items():
        distances["distance_km_" + name] = haversine_km(features["locality_centre_lat"], features["locality_centre_long"],
                                                        landmark_lat, landmark_long)
    return distances


# Sanity checks of the formula on known distances
# 1 degree of latitude is about 111.2 km everywhere
assert abs(haversine_km(17.0, 78.0, 18.0, 78.0) - 111.19) < 0.01
# Symmetric, and zero for the same point
assert haversine_km(17.4, 78.4, 17.3, 78.5) == haversine_km(17.3, 78.5, 17.4, 78.4)
assert haversine_km(17.4, 78.4, 17.4, 78.4) == 0

print("Distances between the landmarks (km):")
landmark_names = list(LANDMARKS)
landmark_distances = pd.DataFrame(index=landmark_names, columns=landmark_names, dtype=float)
for name_1 in landmark_names:
    for name_2 in landmark_names:
        landmark_distances.loc[name_1, name_2] = haversine_km(*LANDMARKS[name_1], *LANDMARKS[name_2])
landmark_distances.round(1)

Distances between the landmarks (km):


,city_centre,airport,hitec_city,gachibowli,financial_district,secunderabad
city_centre,0.0,17.2,13.4,15.9,16.0,5.6
airport,17.2,0.0,23.9,23.8,21.8,22.8
hitec_city,13.4,23.9,0.0,3.6,5.7,12.9
gachibowli,15.9,23.8,3.6,0.0,2.7,16.2
financial_district,16.0,21.8,5.7,2.7,0.0,17.2
secunderabad,5.6,22.8,12.9,16.2,17.2,0.0


**Independent check of the landmark coordinates.** The website's locality table (`localities.csv`) has a centre for
localities with the same names. It contains no prices, so it is a safe cross-check. The landmark coordinates were **not**
taken from it; we only confirm that the two sources agree to within a few km.

In [4]:
# Locality names in the website table that correspond to each landmark (exact names)
MATCHING_LOCALITY_NAME = {
    "city_centre": "Abids",
    "airport": "Shamshabad",
    "hitec_city": "Hitech City",
    "gachibowli": "Gachibowli",
    "financial_district": "Financial District",
    "secunderabad": "Secunderabad",
}

check_rows = []
for name, locality_name in MATCHING_LOCALITY_NAME.items():
    match = locality_table[locality_table["locality_name"] == locality_name].iloc[0]
    gap = haversine_km(*LANDMARKS[name], match["locality_lat"], match["locality_long"])
    check_rows.append({"landmark": name, "website locality": locality_name, "gap (km)": gap})
    assert gap < 3, f"{name}: landmark and website locality are {gap:.1f} km apart"

# Every landmark lies inside the Hyderabad box used since notebook 14
for name, (landmark_lat, landmark_long) in LANDMARKS.items():
    assert HYDERABAD_LAT_RANGE[0] <= landmark_lat <= HYDERABAD_LAT_RANGE[1], name
    assert HYDERABAD_LONG_RANGE[0] <= landmark_long <= HYDERABAD_LONG_RANGE[1], name

print("All landmarks are inside the Hyderabad box and within 3 km of the website's locality of the same name.")
pd.DataFrame(check_rows).round(2)

All landmarks are inside the Hyderabad box and within 3 km of the website's locality of the same name.


,landmark,website locality,gap (km)
0,city_centre,Abids,1.73
1,airport,Shamshabad,0.95
2,hitec_city,Hitech City,0.57
3,gachibowli,Gachibowli,1.04
4,financial_district,Financial District,0.35
5,secunderabad,Secunderabad,0.78


### Listing pins vs locality centres (training set only)

A re-count of notebook 17's comparison, now with the **cleaned** centres from `add_fixed_features()` and the Haversine
distance. It only describes the data; the pins are not used anywhere else.

In [5]:
train_features = add_fixed_features(train, locality_table)

has_pin = train["latitude"].notna() & train["longitude"].notna()
has_centre = train_features["locality_centre_lat"].notna()
pin_inside_box = (train["latitude"].between(*HYDERABAD_LAT_RANGE) & train["longitude"].between(*HYDERABAD_LONG_RANGE))
pin_to_centre_km = haversine_km(train["latitude"], train["longitude"],
                                train_features["locality_centre_lat"], train_features["locality_centre_long"])

pin_counts = pd.Series({
    "training listings": len(train),
    "with a usable locality centre": has_centre.sum(),
    "with a listing pin (not (0,0))": has_pin.sum(),
    "  pin outside the Hyderabad box": (has_pin & ~pin_inside_box).sum(),
    "  pin within 50 m of its locality centre (a copy)": (pin_to_centre_km < 0.05).sum(),
    "  pin more than 20 km from its locality centre": (pin_to_centre_km > 20).sum(),
    "  pin inside the box, differs from the centre, <= 20 km": (has_pin & pin_inside_box & (pin_to_centre_km >= 0.05)
                                                               & (pin_to_centre_km <= 20)).sum(),
    "no centre, but a pin": (has_pin & ~has_centre).sum(),
})
pin_counts.to_frame("listings")

,listings
training listings,16219
with a usable locality centre,15081
"with a listing pin (not (0,0))",10266
pin outside the Hyderabad box,71
pin within 50 m of its locality centre (a copy),4349
pin more than 20 km from its locality centre,259
"pin inside the box, differs from the centre, <= 20 km",5592
"no centre, but a pin",66


**Result:** the locality centre is available for about 93% of training listings and is one checked point per
locality. The pins are missing for over a third of listings, often copy the centre, and are sometimes far off. The
choice of notebooks 17/18 stands: **distances are computed from the locality centre only**. Listings without a usable
centre (mostly "unknown locality") get missing distances, which the pipeline fills with the fold's training median, exactly
like latitude/longitude in the baseline.

## 4. Build the distance features

`add_distance_features()` is a **fixed rule**: each listing's distances depend only on its own locality centre and the six
constants above. Nothing is learned from other listings and no price is read, so it is safe to compute it once for the
whole training set before cross-validation (section 8 also recomputes it inside every fold and checks that the result is
identical).

In [6]:
distance_table = add_distance_features(train_features)
geo_features = pd.concat([train_features, distance_table], axis=1)
y_train = train["price"]
groups = train["property_group_id"]

print("Feature table:", geo_features.shape)
print("Listings without distances (no usable centre):", distance_table.iloc[:, 0].isna().sum())
print()
print("Distance summary (km), listings with a centre:")
distance_table.describe().T[["min", "25%", "50%", "75%", "max"]].round(1)

Feature table: (16219, 27)
Listings without distances (no usable centre): 1138

Distance summary (km), listings with a centre:


,min,25%,50%,75%,max
distance_km_city_centre,0.1,10.2,15.0,20.2,104.3
distance_km_airport,1.0,19.5,26.2,31.4,115.8
distance_km_hitec_city,0.1,7.4,11.3,18.3,99.1
distance_km_gachibowli,0.3,7.3,12.5,20.5,102.1
distance_km_financial_district,0.3,8.4,13.7,21.4,104.7
distance_km_secunderabad,0.4,9.5,14.3,18.1,101.1


**How similar are the geographic columns?** Correlation between latitude, longitude and the six distances (listings with a
centre). Values near ±1 mean two columns carry almost the same information.

In [7]:
GEO_COLUMNS = ["locality_centre_lat", "locality_centre_long"] + DISTANCE_FEATURES
geo_features[GEO_COLUMNS].corr().round(2)

,locality_centre_lat,locality_centre_long,distance_km_city_centre,distance_km_airport,distance_km_hitec_city,distance_km_gachibowli,distance_km_financial_district,distance_km_secunderabad
locality_centre_lat,1.00,-0.19,0.46,0.84,-0.09,-0.06,0.06,0.17
locality_centre_long,-0.19,1.00,-0.34,-0.04,0.55,0.69,0.69,-0.42
distance_km_city_centre,0.46,-0.34,1.00,0.76,0.42,0.28,0.31,0.93
distance_km_airport,0.84,-0.04,0.76,1.00,0.33,0.32,0.42,0.50
distance_km_hitec_city,-0.09,0.55,0.42,0.33,1.00,0.97,0.95,0.41
distance_km_gachibowli,-0.06,0.69,0.28,0.32,0.97,1.00,0.99,0.24
distance_km_financial_district,0.06,0.69,0.31,0.42,0.95,0.99,1.00,0.22
distance_km_secunderabad,0.17,-0.42,0.93,0.50,0.41,0.24,0.22,1.00


## 5. Folds, preprocessing and metrics (identical to notebooks 19–21)

- **Folds:** `GroupKFold(n_splits=5, shuffle=True, random_state=42)`, groups = `property_group_id`, created once.
- **Preprocessing:** the notebook 18/19 `ColumnTransformer`. `make_ridge_model(numeric_features)` takes the list of
  numeric columns, so the four models differ **only** in that list. Every numeric column, including the distances, gets
  median imputation → standard scaling, fitted on each fold's training part.
- **Model:** `Ridge(alpha=1.0)` on log(price) (`TransformedTargetRegressor`), predictions back in rupees.
- **Metrics:** MAE (primary), Median APE (secondary), RMSE, R², per fold, then mean ± sample standard deviation.

In [8]:
RANDOM_STATE = 42
N_FOLDS = 5
cv_splits = list(GroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
                 .split(geo_features, y_train, groups=groups))

expected_sizes = [(12997, 3222), (12980, 3239), (12961, 3258), (12989, 3230), (12949, 3270)]
for fold_number, (train_positions, validation_positions) in enumerate(cv_splits, start=1):
    assert (len(train_positions), len(validation_positions)) == expected_sizes[fold_number - 1]
print("Fold sizes identical to notebooks 19-21.")

BASELINE_NUMERIC = ["bedrooms", "bathrooms", "log_area", "apartment_floor", "apartment_building_floors",
                    "locality_centre_lat", "locality_centre_long"]
LAT_LONG = ["locality_centre_lat", "locality_centre_long"]
NON_GEO_NUMERIC = [column for column in BASELINE_NUMERIC if column not in LAT_LONG]
BINARY_FEATURES = ["area_missing", "area_unit_is_sqyd", "has_land_area_unit", "prime_location_flag", "in_named_project"]
CATEGORICAL_FEATURES = ["property_type", "balconies_band", "transaction_group", "furnishing", "possession_status",
                        "age_of_property", "facing_clean", "posted_by"]
LOCALITY_FEATURES = ["locality_code"]
MIN_LOCALITY_LISTINGS = 10
RIDGE_ALPHA = 1.0
assert sorted(BASELINE_NUMERIC + BINARY_FEATURES + CATEGORICAL_FEATURES + LOCALITY_FEATURES) == sorted(train_features.columns)

# The numeric columns of each model; everything else is shared
MODELS = {
    "A. Baseline (lat/long)": BASELINE_NUMERIC,
    "B. No coordinates": NON_GEO_NUMERIC,
    "C. Distances instead of lat/long": NON_GEO_NUMERIC + DISTANCE_FEATURES,
    "D. Lat/long + distances": BASELINE_NUMERIC + DISTANCE_FEATURES,
}
for name, numeric_features in MODELS.items():
    print(f"{name:35} numeric columns: {numeric_features}")


def make_ridge_model(numeric_features):
    preprocessor = ColumnTransformer(
        transformers=[
            ("numeric", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]),
             numeric_features),
            ("binary", "passthrough", BINARY_FEATURES),
            ("categorical", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CATEGORICAL_FEATURES),
            ("locality", OneHotEncoder(min_frequency=MIN_LOCALITY_LISTINGS, handle_unknown="infrequent_if_exist",
                                       sparse_output=False), LOCALITY_FEATURES),
        ]
    )
    return Pipeline([
        ("preprocess", preprocessor),
        ("ridge", TransformedTargetRegressor(regressor=Ridge(alpha=RIDGE_ALPHA), func=np.log, inverse_func=np.exp)),
    ])

Fold sizes identical to notebooks 19-21.
A. Baseline (lat/long)              numeric columns: ['bedrooms', 'bathrooms', 'log_area', 'apartment_floor', 'apartment_building_floors', 'locality_centre_lat', 'locality_centre_long']
B. No coordinates                   numeric columns: ['bedrooms', 'bathrooms', 'log_area', 'apartment_floor', 'apartment_building_floors']
C. Distances instead of lat/long    numeric columns: ['bedrooms', 'bathrooms', 'log_area', 'apartment_floor', 'apartment_building_floors', 'distance_km_city_centre', 'distance_km_airport', 'distance_km_hitec_city', 'distance_km_gachibowli', 'distance_km_financial_district', 'distance_km_secunderabad']
D. Lat/long + distances             numeric columns: ['bedrooms', 'bathrooms', 'log_area', 'apartment_floor', 'apartment_building_floors', 'locality_centre_lat', 'locality_centre_long', 'distance_km_city_centre', 'distance_km_airport', 'distance_km_hitec_city', 'distance_km_gachibowli', 'distance_km_financial_district', 'distance

In [9]:
def median_ape(actual, predicted):
    return np.median(np.abs(predicted - actual) / actual) * 100


def run_cv(numeric_features):
    # The same fold loop as notebooks 20-21: a fresh pipeline per fold, fitted on the training part only
    model = make_ridge_model(numeric_features)
    fold_rows = []
    out_of_fold = pd.Series(np.nan, index=geo_features.index)
    fold_details = []

    for fold_number, (train_positions, validation_positions) in enumerate(cv_splits, start=1):
        fold_model = clone(model)
        fold_model.fit(geo_features.iloc[train_positions], y_train.iloc[train_positions])
        actual = y_train.iloc[validation_positions]
        predicted = fold_model.predict(geo_features.iloc[validation_positions])
        out_of_fold.iloc[validation_positions] = predicted

        fold_rows.append({
            "fold": fold_number,
            "MAE": mean_absolute_error(actual, predicted),
            "RMSE": root_mean_squared_error(actual, predicted),
            "R2": r2_score(actual, predicted),
            "Median APE": median_ape(actual, predicted),
        })
        imputer = fold_model.named_steps["preprocess"].named_transformers_["numeric"].named_steps["impute"]
        fold_details.append({
            "imputer_medians": imputer.statistics_.copy(),
            "expected_medians": geo_features.iloc[train_positions][numeric_features].median().values,
            "prediction_count": len(predicted),
            "fitted_model": fold_model,
        })
    return pd.DataFrame(fold_rows), out_of_fold, fold_details


def summarise(model_name, fold_table):
    return {
        "Model": model_name,
        "MAE (₹ lakh)": fold_table["MAE"].mean() / 1e5,
        "MAE std": fold_table["MAE"].std() / 1e5,
        "Median APE (%)": fold_table["Median APE"].mean(),
        "Median APE std": fold_table["Median APE"].std(),
        "RMSE (₹ lakh)": fold_table["RMSE"].mean() / 1e5,
        "RMSE std": fold_table["RMSE"].std() / 1e5,
        "R²": fold_table["R2"].mean(),
        "R² std": fold_table["R2"].std(),
    }

## 6. Compare the four representations

First model A, checked against notebook 19; then B, C and D on the same folds.

In [10]:
fold_tables = {}
out_of_fold_predictions = {}
fold_details = {}
for name, numeric_features in MODELS.items():
    fold_tables[name], out_of_fold_predictions[name], fold_details[name] = run_cv(numeric_features)

baseline = fold_tables["A. Baseline (lat/long)"]
# The values printed by notebooks 19 and 20 (rounded to 3 decimals there)
assert np.isclose(baseline["MAE"].mean() / 1e5, 70.248, atol=0.001)
assert np.isclose(baseline["RMSE"].mean() / 1e5, 179.528, atol=0.001)
assert np.isclose(baseline["R2"].mean(), 0.481, atol=0.001)
assert np.isclose(baseline["Median APE"].mean(), 21.841, atol=0.001)
print("Model A reproduces notebook 19's Ridge: MAE 70.248, RMSE 179.528, R² 0.481, Median APE 21.841.")
print()
summary = pd.DataFrame([summarise(name, fold_tables[name]) for name in MODELS])
summary.round(3)

Model A reproduces notebook 19's Ridge: MAE 70.248, RMSE 179.528, R² 0.481, Median APE 21.841.



,Model,MAE (₹ lakh),MAE std,Median APE (%),Median APE std,RMSE (₹ lakh),RMSE std,R²,R² std
0,A. Baseline (lat/long),70.248,3.105,21.841,0.204,179.528,20.846,0.481,0.028
1,B. No coordinates,70.410,3.054,22.007,0.370,179.948,20.930,0.478,0.027
2,C. Distances instead of lat/long,69.365,3.199,21.372,0.430,177.689,20.734,0.491,0.029
3,D. Lat/long + distances,69.320,3.210,21.385,0.412,177.667,20.729,0.491,0.029


### Per-fold values

In [11]:
for metric, scale, label in [("MAE", 1e5, "MAE (₹ lakh)"), ("Median APE", 1, "Median APE (%)"),
                             ("RMSE", 1e5, "RMSE (₹ lakh)"), ("R2", 1, "R²")]:
    table = pd.DataFrame({name: fold_tables[name][metric].values / scale for name in MODELS},
                         index=pd.Index(range(1, 6), name="fold"))
    print(label)
    print(table.round(3).to_string())
    print()

MAE (₹ lakh)
      A. Baseline (lat/long)  B. No coordinates  C. Distances instead of lat/long  D. Lat/long + distances
fold                                                                                                      
1                     67.217             67.525                            66.365                   66.312
2                     67.066             67.285                            65.912                   65.841
3                     73.583             73.736                            72.874                   72.848
4                     70.270             70.245                            69.542                   69.527
5                     73.102             73.259                            72.134                   72.071

Median APE (%)
      A. Baseline (lat/long)  B. No coordinates  C. Distances instead of lat/long  D. Lat/long + distances
fold                                                                                                      
1       

### Change against the baseline, fold by fold

For B, C and D: the change from A in each fold, and in how many folds the model is better. Negative MAE / Median APE /
RMSE changes and positive R² changes mean better than A.

In [12]:
def change_against_baseline(fold_table):
    return pd.DataFrame({
        "MAE change (₹ lakh)": (fold_table["MAE"] - baseline["MAE"]).values / 1e5,
        "MAE change (%)": ((fold_table["MAE"] - baseline["MAE"]) / baseline["MAE"]).values * 100,
        "Median APE change (points)": (fold_table["Median APE"] - baseline["Median APE"]).values,
        "RMSE change (₹ lakh)": (fold_table["RMSE"] - baseline["RMSE"]).values / 1e5,
        "R² change": (fold_table["R2"] - baseline["R2"]).values,
    }, index=pd.Index(range(1, 6), name="fold"))


consistency_rows = []
for name in list(MODELS)[1:]:
    change = change_against_baseline(fold_tables[name])
    print(name)
    print(change.round(3).to_string())
    print()
    consistency_rows.append({
        "Model": name,
        "mean MAE change (%)": change["MAE change (%)"].mean(),
        "folds with lower MAE": (change["MAE change (₹ lakh)"] < 0).sum(),
        "mean Median APE change": change["Median APE change (points)"].mean(),
        "folds with lower Median APE": (change["Median APE change (points)"] < 0).sum(),
        "folds with lower RMSE": (change["RMSE change (₹ lakh)"] < 0).sum(),
        "folds with higher R²": (change["R² change"] > 0).sum(),
    })
consistency = pd.DataFrame(consistency_rows).set_index("Model")
consistency.round(3)

B. No coordinates
      MAE change (₹ lakh)  MAE change (%)  Median APE change (points)  RMSE change (₹ lakh)  R² change
fold                                                                                                  
1                   0.308           0.458                       0.180                 0.484     -0.003
2                   0.218           0.325                      -0.166                 0.731     -0.004
3                   0.153           0.209                       0.183                 0.067     -0.000
4                  -0.025          -0.036                       0.516                 0.048     -0.000
5                   0.157           0.215                       0.117                 0.773     -0.004

C. Distances instead of lat/long
      MAE change (₹ lakh)  MAE change (%)  Median APE change (points)  RMSE change (₹ lakh)  R² change
fold                                                                                                  
1                  -0

,mean MAE change (%),folds with lower MAE,mean Median APE change,folds with lower Median APE,folds with lower RMSE,folds with higher R²
Model,,,,,,
B. No coordinates,0.234,1,0.166,1,0,0
C. Distances instead of lat/long,-1.263,5,-0.469,5,5,5
D. Lat/long + distances,-1.328,5,-0.456,5,5,5


## 7. One distance at a time

Each distance is added **alone** to the baseline (A + one distance), on the same folds. A distance that does not
improve the baseline on its own, or whose gain is entirely covered by a nearby landmark, is a candidate for rejection.
This is a diagnostic, not a search for the best subset: choosing the best-looking subset from many combinations
would over-fit to these folds.

In [13]:
single_rows = []
single_fold_tables = {}
for distance in DISTANCE_FEATURES:
    fold_table, _, _ = run_cv(BASELINE_NUMERIC + [distance])
    single_fold_tables[distance] = fold_table
    change = change_against_baseline(fold_table)
    single_rows.append({
        "A + ...": distance,
        "MAE (₹ lakh)": fold_table["MAE"].mean() / 1e5,
        "MAE change (%)": change["MAE change (%)"].mean(),
        "folds with lower MAE": (change["MAE change (₹ lakh)"] < 0).sum(),
        "Median APE (%)": fold_table["Median APE"].mean(),
        "Median APE change": change["Median APE change (points)"].mean(),
        "folds with lower Median APE": (change["Median APE change (points)"] < 0).sum(),
        "RMSE change (₹ lakh)": change["RMSE change (₹ lakh)"].mean(),
        "R² change": change["R² change"].mean(),
    })
pd.DataFrame(single_rows).set_index("A + ...").round(3)

,MAE (₹ lakh),MAE change (%),folds with lower MAE,Median APE (%),Median APE change,folds with lower Median APE,RMSE change (₹ lakh),R² change
A + ...,,,,,,,,
distance_km_city_centre,69.688,-0.800,5,21.576,-0.265,5,-0.901,0.005
distance_km_airport,69.849,-0.569,5,21.632,-0.209,5,-0.717,0.004
distance_km_hitec_city,69.585,-0.947,5,21.539,-0.302,5,-1.174,0.007
distance_km_gachibowli,69.593,-0.936,5,21.499,-0.342,5,-1.179,0.007
distance_km_financial_district,69.568,-0.971,5,21.548,-0.292,5,-1.291,0.007
distance_km_secunderabad,69.759,-0.699,5,21.619,-0.221,5,-0.760,0.004


**Ridge coefficients of the geographic columns in model D.** Ridge learns log(price) and the columns are standardised,
so a coefficient of −0.1 means one standard deviation further away → about 10% lower predicted price, **holding the
other columns fixed**. With strongly correlated distances, individual coefficients can split, cancel or flip sign.
They are shown to see whether the model uses the columns in a stable way across folds, not to interpret each one.

In [14]:
coefficient_table = {}
for fold_number, detail in enumerate(fold_details["D. Lat/long + distances"], start=1):
    fitted = detail["fitted_model"]
    names = fitted.named_steps["preprocess"].get_feature_names_out()
    coefficients = pd.Series(fitted.named_steps["ridge"].regressor_.coef_, index=names)
    coefficient_table[f"fold {fold_number}"] = coefficients[["numeric__" + column for column in GEO_COLUMNS]].values
pd.DataFrame(coefficient_table, index=GEO_COLUMNS).round(3)

,fold 1,fold 2,fold 3,fold 4,fold 5
locality_centre_lat,-0.063,-0.045,-0.050,-0.051,-0.067
locality_centre_long,-0.001,-0.010,0.006,-0.010,-0.016
distance_km_city_centre,-0.227,-0.251,-0.230,-0.258,-0.215
distance_km_airport,0.203,0.209,0.211,0.214,0.209
distance_km_hitec_city,-0.436,-0.339,-0.403,-0.385,-0.380
distance_km_gachibowli,0.711,0.703,0.794,0.681,0.707
distance_km_financial_district,-0.458,-0.535,-0.581,-0.475,-0.496
distance_km_secunderabad,0.210,0.197,0.199,0.213,0.171


## 8. Where the effect appears

Out-of-fold predictions (each listing predicted by the model of the fold in which it was validated). For each group:
MAE and Median APE for every model, and the change from A.

- **Price band** is derived from the target and is used **only for reporting**.
- **Locality availability:** whether the listing has a usable locality centre (and therefore real distances), and whether
  its locality is a **frequent** one (≥ 10 listings in its fold's training part; own one-hot column), a **rare/unseen**
  one (shares the "rare" column) or **unknown**. Listings without a centre receive the fold median for every distance, so
  no gain is expected there.

In [15]:
RANGE_NAMES = ["0-50 L", "50-80 L", "80 L-1.2 Cr", "1.2-1.5 Cr", "1.5-1.75 Cr",
               "1.75-2 Cr", "2-3 Cr", "3-5 Cr", "5 Cr+"]
PROPERTY_TYPES = ["Apartment", "Builder Floor Apartment", "Penthouse", "Residential House", "Villa"]
MODEL_NAMES = list(MODELS)
SHORT = {name: name.split(".")[0] for name in MODEL_NAMES}   # "A", "B", "C", "D"

# Locality status of every listing, in the fold where it is validated
locality_status = pd.Series("", index=geo_features.index)
for train_positions, validation_positions in cv_splits:
    training_counts = geo_features["locality_code"].iloc[train_positions].value_counts()
    validation_localities = geo_features["locality_code"].iloc[validation_positions]
    count_in_training = validation_localities.map(training_counts).fillna(0)
    status = pd.Series("rare or unseen locality", index=validation_localities.index)
    status[count_in_training >= MIN_LOCALITY_LISTINGS] = "frequent locality"
    status[validation_localities == "unknown locality"] = "unknown locality"
    locality_status[validation_localities.index] = status
assert (locality_status != "").all()

errors = pd.DataFrame({
    "price": y_train,
    "price_band": train["price_band"],
    "property_type": train["property_type"],
    "type_family": np.where(train["property_type"].isin(["Residential House", "Villa"]), "house / villa",
                            "apartment-like"),
    "locality_status": locality_status,
    "centre": np.where(geo_features["locality_centre_lat"].notna(), "has centre", "no centre"),
}, index=train.index)
for name in MODEL_NAMES:
    errors[SHORT[name] + " abs error"] = (out_of_fold_predictions[name] - y_train).abs()
    errors[SHORT[name] + " abs % error"] = errors[SHORT[name] + " abs error"] / y_train * 100


def compare_by(group_column, order):
    grouped = errors.groupby(group_column)
    table = pd.DataFrame({"listings": grouped.size()})
    for name in MODEL_NAMES:
        table[SHORT[name] + " MAE"] = grouped[SHORT[name] + " abs error"].mean() / 1e5
    for name in MODEL_NAMES[1:]:
        table[SHORT[name] + "−A MAE"] = table[SHORT[name] + " MAE"] - table["A MAE"]
    for name in MODEL_NAMES:
        table[SHORT[name] + " APE"] = grouped[SHORT[name] + " abs % error"].median()
    for name in MODEL_NAMES[1:]:
        table[SHORT[name] + "−A APE"] = table[SHORT[name] + " APE"] - table["A APE"]
    return table.reindex(order).round(1)


print("MAE in ₹ lakh, APE = median absolute % error; '−A' columns = change from the baseline")

MAE in ₹ lakh, APE = median absolute % error; '−A' columns = change from the baseline


### By property type

In [16]:
compare_by("property_type", PROPERTY_TYPES)

,listings,A MAE,B MAE,C MAE,D MAE,B−A MAE,C−A MAE,D−A MAE,A APE,B APE,C APE,D APE,B−A APE,C−A APE,D−A APE
property_type,,,,,,,,,,,,,,,
Apartment,9218,39.2,39.3,38.8,38.8,0.1,-0.3,-0.4,17.5,17.6,17.3,17.2,0.1,-0.2,-0.2
Builder Floor Apartment,498,34.0,33.9,33.7,33.7,-0.0,-0.3,-0.3,22.2,22.8,22.3,22.4,0.6,0.2,0.2
Penthouse,34,41.0,40.3,44.3,44.1,-0.7,3.4,3.2,20.4,20.0,18.1,17.8,-0.4,-2.3,-2.5
Residential House,4353,99.5,100.0,98.5,98.4,0.5,-1.0,-1.1,31.8,32.3,31.3,31.2,0.6,-0.4,-0.5
Villa,2116,154.6,154.2,151.5,151.4,-0.4,-3.2,-3.3,29.1,28.9,27.9,27.9,-0.2,-1.2,-1.1


In [17]:
compare_by("type_family", ["apartment-like", "house / villa"])

,listings,A MAE,B MAE,C MAE,D MAE,B−A MAE,C−A MAE,D−A MAE,A APE,B APE,C APE,D APE,B−A APE,C−A APE,D−A APE
type_family,,,,,,,,,,,,,,,
apartment-like,9750,38.9,39.0,38.6,38.5,0.1,-0.3,-0.4,17.7,17.9,17.5,17.5,0.1,-0.2,-0.2
house / villa,6469,117.5,117.8,115.8,115.7,0.2,-1.7,-1.8,30.8,31.0,30.2,30.1,0.3,-0.6,-0.6


### By price band

In [18]:
compare_by("price_band", RANGE_NAMES)

,listings,A MAE,B MAE,C MAE,D MAE,B−A MAE,C−A MAE,D−A MAE,A APE,B APE,C APE,D APE,B−A APE,C−A APE,D−A APE
price_band,,,,,,,,,,,,,,,
0-50 L,1260,27.0,27.2,26.4,26.3,0.2,-0.6,-0.7,52.1,52.4,50.2,50.3,0.2,-1.9,-1.8
50-80 L,2291,20.7,21.0,20.8,20.8,0.3,0.1,0.1,19.7,20.4,19.9,19.9,0.7,0.2,0.2
80 L-1.2 Cr,2402,23.8,24.1,23.7,23.7,0.3,-0.1,-0.1,15.2,15.6,15.4,15.4,0.4,0.2,0.2
1.2-1.5 Cr,2287,34.8,34.9,34.3,34.3,0.0,-0.6,-0.6,17.5,17.7,17.5,17.4,0.1,-0.0,-0.1
1.5-1.75 Cr,1687,41.6,41.6,41.2,41.2,-0.0,-0.4,-0.5,18.9,19.0,18.7,18.6,0.0,-0.2,-0.3
1.75-2 Cr,1616,50.4,50.4,49.2,49.1,-0.0,-1.2,-1.3,20.1,20.3,19.0,18.9,0.2,-1.1,-1.2
2-3 Cr,2162,71.4,71.6,70.3,70.2,0.2,-1.1,-1.2,21.7,21.8,20.8,20.8,0.1,-0.9,-0.9
3-5 Cr,1550,131.4,131.7,130.7,130.7,0.3,-0.7,-0.7,29.0,29.1,29.1,29.0,0.0,0.1,0.0
5 Cr+,964,426.9,427.1,420.5,420.5,0.2,-6.4,-6.4,37.1,37.6,36.3,36.6,0.4,-0.8,-0.6


### By locality availability

In [19]:
compare_by("centre", ["has centre", "no centre"])

,listings,A MAE,B MAE,C MAE,D MAE,B−A MAE,C−A MAE,D−A MAE,A APE,B APE,C APE,D APE,B−A APE,C−A APE,D−A APE
centre,,,,,,,,,,,,,,,
has centre,15081,69.8,70.0,68.8,68.8,0.2,-1.0,-1.0,21.3,21.4,20.8,20.8,0.1,-0.5,-0.5
no centre,1138,76.6,76.5,77.0,77.0,-0.1,0.4,0.5,29.7,29.6,30.6,30.6,-0.1,0.8,0.8


In [20]:
compare_by("locality_status", ["frequent locality", "rare or unseen locality", "unknown locality"])

,listings,A MAE,B MAE,C MAE,D MAE,B−A MAE,C−A MAE,D−A MAE,A APE,B APE,C APE,D APE,B−A APE,C−A APE,D−A APE
locality_status,,,,,,,,,,,,,,,
frequent locality,10779,65.9,65.9,65.6,65.6,0.0,-0.3,-0.4,18.9,18.8,18.7,18.7,-0.0,-0.1,-0.2
rare or unseen locality,4350,79.4,80.0,76.9,76.7,0.6,-2.5,-2.7,28.8,29.5,27.5,27.2,0.7,-1.3,-1.6
unknown locality,1090,76.7,76.7,77.2,77.2,-0.1,0.4,0.5,29.6,29.4,30.4,30.4,-0.2,0.8,0.8


In [21]:
# Apartment-like vs house/villa within each locality status: where do C and D help or hurt?
errors["family_and_status"] = errors["type_family"] + " | " + errors["locality_status"]
order = [family + " | " + status for family in ["apartment-like", "house / villa"]
         for status in ["frequent locality", "rare or unseen locality", "unknown locality"]]
compare_by("family_and_status", order)

,listings,A MAE,B MAE,C MAE,D MAE,B−A MAE,C−A MAE,D−A MAE,A APE,B APE,C APE,D APE,B−A APE,C−A APE,D−A APE
family_and_status,,,,,,,,,,,,,,,
apartment-like | frequent locality,7188,37.5,37.5,37.5,37.5,-0.1,-0.1,-0.1,15.9,16.0,15.8,15.8,0.0,-0.1,-0.1
apartment-like | rare or unseen locality,1982,40.0,40.9,38.6,38.5,0.8,-1.5,-1.6,22.7,23.6,21.9,22.0,0.8,-0.8,-0.7
apartment-like | unknown locality,580,51.8,51.6,52.1,52.1,-0.2,0.3,0.3,25.2,25.1,25.5,25.4,-0.1,0.3,0.3
house / villa | frequent locality,3591,122.7,122.9,121.8,121.8,0.2,-0.9,-0.9,27.2,27.3,27.3,27.2,0.2,0.1,0.0
house / villa | rare or unseen locality,2368,112.3,112.7,108.9,108.7,0.4,-3.4,-3.7,36.0,35.8,34.1,33.8,-0.1,-1.8,-2.2
house / villa | unknown locality,510,105.1,105.2,105.7,105.7,0.1,0.6,0.6,37.2,37.4,37.6,37.7,0.2,0.4,0.5


## 9. Leakage and reproducibility checks

Each check is an `assert`.

1. **No test rows were used.**
2. **No target-derived geographic feature:** the distances are computed from the two centre columns only; recomputing
   them with every price replaced by random numbers (and with the price columns removed) gives identical values; no
   forbidden column is among the inputs.
3. **The landmark coordinates are independent of the target:** they are six constants typed into the notebook (section
   3), defined before any price was read into a model; recomputing distances from a copy of the data without any price
   column gives the same result.
4. **Transformations are reproduced identically inside each fold:** computing the distances from each fold's training
   and validation rows separately gives exactly the values computed once for the whole training set; each fold's imputer
   medians equal that fold's training medians and differ between folds.
5. **Property groups never cross fold boundaries.**
6. **The baseline reproduces notebook 19** (asserted in section 6).
7. **Predictions are valid**: no NaN/infinite/non-positive values, one prediction per validation row, every training
   row validated exactly once.

In [22]:
# 1. No test rows
assert set(train["property_id"]).isdisjoint(test_property_ids)
assert (train["split"] == "train").all() and len(geo_features) == 16219
print("1. None of the", test_row_count, "test listings was used; only the 16,219 training listings.")

# 2. Distances are not target-derived
FORBIDDEN_COLUMNS = ["price", "price_band", "price_per_sqft", "loc_avg_price_sqft", "loc_avg_price_lmt",
                     "loc_price_quarter_prev", "loc_price_quarter_current", "loc_growth_quarterly_pct",
                     "loc_locality_rank"]
for column in FORBIDDEN_COLUMNS:
    assert column not in INPUT_COLUMNS and column not in geo_features.columns, column

scrambled_train = train.copy()
scrambled_train["price"] = np.random.default_rng(RANDOM_STATE).uniform(1e5, 1e10, len(train))
scrambled_distances = add_distance_features(add_fixed_features(scrambled_train, locality_table))
assert scrambled_distances.equals(distance_table)

price_columns = [column for column in train.columns if "price" in column]
no_price_train = train.drop(columns=price_columns)
no_price_distances = add_distance_features(add_fixed_features(no_price_train, locality_table))
assert no_price_distances.equals(distance_table)
print("2. No forbidden column among the inputs; distances are identical with random prices and with all",
      len(price_columns), "price columns removed.")

# 3. Landmarks are constants, not data
for name, coordinates in LANDMARKS.items():
    assert isinstance(coordinates, tuple) and all(isinstance(value, float) for value in coordinates), name
print("3. The six landmarks are fixed constants; distances do not change when prices are removed (check 2).")

1. None of the 4081 test listings was used; only the 16,219 training listings.


2. No forbidden column among the inputs; distances are identical with random prices and with all 2 price columns removed.
3. The six landmarks are fixed constants; distances do not change when prices are removed (check 2).


In [23]:
# 4. Transformations reproduced identically inside each fold
for fold_number, (train_positions, validation_positions) in enumerate(cv_splits, start=1):
    for positions in [train_positions, validation_positions]:
        fold_rows = train.iloc[positions]
        fold_distances = add_distance_features(add_fixed_features(fold_rows, locality_table))
        assert fold_distances.equals(distance_table.iloc[positions]), f"fold {fold_number}"

for name, details in fold_details.items():
    for detail in details:
        assert np.allclose(detail["imputer_medians"], detail["expected_medians"]), name
    assert not all(np.allclose(details[0]["imputer_medians"], detail["imputer_medians"]) for detail in details[1:]), name
print("4. Fixed features computed per fold = computed once; each fold's imputer medians = its training medians,")
print("   and they differ between folds (for every model).")

# 5. Groups never cross fold boundaries
for fold_number, (train_positions, validation_positions) in enumerate(cv_splits, start=1):
    assert set(groups.iloc[train_positions]).isdisjoint(set(groups.iloc[validation_positions])), fold_number
print("5. No property_group_id on both sides of any fold.")

# 6. Baseline reproduction was asserted in section 6
print("6. Model A reproduces notebook 19 (asserted in section 6).")

# 7. Valid predictions
validation_counts = np.zeros(len(geo_features), dtype=int)
for train_positions, validation_positions in cv_splits:
    validation_counts[validation_positions] += 1
assert (validation_counts == 1).all()
for name in MODEL_NAMES:
    predictions = out_of_fold_predictions[name]
    assert predictions.notna().all() and np.isfinite(predictions).all() and (predictions > 0).all(), name
    assert predictions.index.equals(geo_features.index)
    for detail, (train_positions, validation_positions) in zip(fold_details[name], cv_splits):
        assert detail["prediction_count"] == len(validation_positions)
print("7. No NaN/infinite/non-positive predictions; one per validation row; each training row validated once.")

4. Fixed features computed per fold = computed once; each fold's imputer medians = its training medians,
   and they differ between folds (for every model).
5. No property_group_id on both sides of any fold.
6. Model A reproduces notebook 19 (asserted in section 6).
7. No NaN/infinite/non-positive predictions; one per validation row; each training row validated once.


## 10. Findings and Decision

All results are 5-fold `GroupKFold` cross-validation on the 16,219 training listings (the same folds as notebooks
19–21). Model A reproduced notebook 19 exactly, every check in section 9 passed, and the test set was not used.

| Model | MAE (₹ lakh) | Median APE | RMSE (₹ lakh) | R² | MAE change vs A | Folds better (MAE / Median APE) |
|---|---|---|---|---|---|---|
| A. Baseline (lat/long) | 70.25 ± 3.11 | 21.84 ± 0.20% | 179.53 ± 20.85 | 0.481 ± 0.028 | – | – |
| B. No coordinates | 70.41 ± 3.05 | 22.01 ± 0.37% | 179.95 ± 20.93 | 0.478 ± 0.027 | +0.2% | 1 / 1 of 5 |
| C. Distances instead of lat/long | 69.37 ± 3.20 | 21.37 ± 0.43% | 177.69 ± 20.73 | 0.491 ± 0.029 | −1.26% | 5 / 5 of 5 |
| D. Lat/long + distances | 69.32 ± 3.21 | 21.39 ± 0.41% | 177.67 ± 20.73 | 0.491 ± 0.029 | −1.33% | 5 / 5 of 5 |

### 1. Do geographic features improve Ridge?

**Yes, modestly and consistently.** Replacing latitude/longitude with the six distances (C) lowers mean CV MAE by
₹0.88 lakh (−1.26%) and Median APE by 0.47 points. MAE, Median APE, RMSE and R² are all better **in every one of the 5
folds**. The MAE gain per fold is 0.96–1.72%. The Median APE gain is clear in folds 1–3 (−0.43 to −0.85 points) but small
in folds 4 and 5 (−0.19 and −0.31).

The gain is small next to the fold-to-fold spread of MAE (std ≈ ₹3.1 lakh), but because every model uses the same
folds, the fact that it goes the same way in all five folds is the stronger evidence.

### 2. Which representation is useful: lat/long, distances, or both?

- **Latitude/longitude on their own contribute very little.** Removing them (B) costs only +0.2% MAE and +0.17 points
  of Median APE, and B is worse in 4 of 5 folds by small amounts. A linear term in latitude or longitude cannot express
  "prices fall away from the IT corridor in every direction".
- **Distances are the useful representation.** C is clearly better than A.
- **Both (D) adds nothing over distances alone.** D vs C: MAE −₹0.045 lakh, Median APE +0.01 points, and the same RMSE
  and R². In D, the longitude coefficient is almost zero in every fold (−0.016 to +0.006) and latitude is small (≈ −0.05).

**The useful representation is distances only (C).**

### 3. Is the improvement consistent enough to justify adding the features?

The decision rule was written down **before** the results (section 1):

| Criterion | C (distances instead of lat/long) | D (lat/long + distances) |
|---|---|---|
| mean MAE at least 1% lower than A | yes (−1.26%) | yes (−1.33%) |
| MAE and Median APE lower in ≥ 4 of 5 folds | yes (5 and 5) | yes (5 and 5) |
| no type family worse by > 0.5 points Median APE | yes (apartment-like −0.2, house/villa −0.6) | yes (−0.2, −0.6) |

Both pass, **C only narrowly** on the 1% threshold (−1.26%; fold 3 alone is −0.96%). The added complexity is low: six
columns from a fixed formula and six constants, nothing learned from prices, no fold-specific fitting beyond the
existing imputation and scaling. This is a different situation from the locality target encoding of notebook 21 (−0.5%,
which needed its own group-aware cross-fitting).

**Decision: recommend C, the six distances *replacing* the two centre coordinates, as the geographic representation for
the final feature set.** Choose C over D because D's extra gain is negligible and C has two fewer columns. The feature is
**not** added to `build_features.py` in this notebook. That is a separate, explicit step.

This is a moderate gain, not a breakthrough. The recommended Ridge (MAE ₹69.4 lakh, Median APE 21.4%) is still behind
notebook 20's Random Forest on typical listings (₹68.6 lakh, 20.2%), though it remains better on RMSE and R².

### 4. Are the effects different for apartments vs houses/villas?

Yes: **houses and villas gain more.**

- Apartment-like types: MAE −₹0.3 lakh, Median APE 17.7% → 17.5%.
- Houses and villas: MAE −₹1.7 lakh, Median APE 30.8% → 30.2% (Villas 29.1% → 27.9%; Residential Houses 31.8% → 31.3%).
- The biggest single group gain is **houses and villas in rare or unseen localities** (2,368 listings): Median APE
  36.0% → 34.1% (C). Apartments in rare/unseen localities also improve (22.7% → 21.9%).
- In **frequent localities** almost nothing changes (Median APE −0.1 points), because their own one-hot column already
  carries the location.
- Builder Floor Apartments get very slightly worse (+0.2 points). Penthouses (34 listings) are too few to judge.

By price band, C helps most in ₹0–50 L (Median APE −1.9 points), ₹1.75–3 Cr (−0.9 to −1.1) and ₹5 Cr+ (MAE −₹6.4 lakh).
₹50 L–1.2 Cr gets very slightly worse (+0.2 points). The "pull toward the middle" of notebook 19 remains.

**Listings without a usable centre** (1,138, almost all "unknown locality") get slightly **worse** (Median APE 29.7% →
30.6%). They receive the fold's median distance for every landmark, an "average location" that is not a real place.
A possible reason (not tested) is that the distance coefficients are larger than the old lat/long coefficients, so
this made-up average location now moves their predictions more. This is a known cost of the recommendation, affecting
about 7% of listings.

### 5. Should any geographic feature be rejected?

- **Reject latitude/longitude as numeric features** once the distances are used (question 2): they are redundant with
  the distances and contribute almost nothing on their own.
- **Listing pins stay rejected** (section 2): only about a third add plausible detail.
- **No single distance is rejected.** Each of the six improves the baseline on its own in all 5 folds (MAE −0.57% for
  the airport to −0.97% for the Financial District). None is useless.
- **The three IT-corridor distances are strongly redundant** (HITEC City, Gachibowli and Financial District correlate
  0.95–0.99; the landmarks are 2.7–5.7 km apart). In model D Ridge gives them large coefficients of **opposite sign**
  (Gachibowli ≈ +0.7, HITEC City ≈ −0.4, Financial District ≈ −0.5). The signs are stable across folds, so this is a
  consistent "contrast" inside the corridor, not noise. But the individual coefficients cannot be interpreted as "the
  effect of being near Gachibowli". Similarly, city centre and Secunderabad correlate 0.93.
- Dropping some of the corridor distances might keep the gain with fewer columns. Choosing a subset **after** seeing
  these results would tune the feature set to these folds, so it is **not** done here. If a smaller set is wanted, it
  must be written down in advance and tested once on the same folds (for example: city centre, airport, one IT-corridor
  point, Secunderabad).

**Limitations.** Asking prices, not sale prices; a non-random sample (price-range searches); straight-line distance, not
travel time; landmark coordinates approximate to about 0.5–1 km; Ridge only; CV results on the training set, not a
test-set evaluation.

## What this notebook establishes

- **Locality-centre coordinates, not listing pins,** are the reliable geographic input (93% coverage, one checked point
  per locality).
- **Six fixed-landmark Haversine distances** (city centre, RGIA airport, HITEC City, Gachibowli, Financial District,
  Secunderabad), computed from the locality centre, improve Ridge by about 1.3% MAE and 0.5 points Median APE, in all 5
  folds and on all four metrics. The gain is concentrated in rare/unseen localities and in houses/villas.
- **Linear latitude/longitude are nearly useless** for Ridge and become redundant once distances are present.
- The recommended geographic representation is **distances replacing latitude/longitude** (model C). It meets the
  pre-set decision rule, narrowly. Locality target encoding stays out (notebook 21).

**Next modelling step (defined, not implemented):**

1. Make the geographic change an explicit, documented update of the feature pipeline: add the six distances as a fixed
   rule in `build_features.py` (landmark constants and `haversine_km`), and drop the centre lat/long from Ridge's numeric
   inputs. Then re-run notebook 19's baseline checks on the new feature set so that there is one new reference Ridge.
2. With that feature set, compare **Ridge against tree-based models** (the fixed Random Forest of notebook 20 and a
   fixed-configuration `HistGradientBoostingRegressor`) on the **same folds and metrics**. Trees may use latitude/
   longitude differently from Ridge (they can split on them), so for trees the "lat/long vs distances" question should
   be checked again, not assumed.
3. Before choosing a final model, decide the **primary metric** (rupee MAE vs Median APE vs RMSE), because Ridge and the
   Random Forest rank differently on them.
4. Keep the unknown-locality listings in mind: they get slightly worse with distances and remain one of the weakest
   groups.

## 11. Reproducibility record

| Item | Value |
|---|---|
| Rows used | the 16,219 training listings; the 4,081 test listings were removed from memory before any modelling |
| Cross-validation | `GroupKFold(n_splits=5, shuffle=True, random_state=42)`, `groups = property_group_id`; the same folds as notebooks 19–21 |
| Target | `price` (asking price); Ridge learns `log(price)`; metrics in rupees |
| Model | `Ridge(alpha=1.0)`, not tuned |
| Shared features | the 19 non-coordinate features of `add_fixed_features()` (notebook 18) |
| Geographic columns | A: centre lat/long; B: none; C: 6 distances; D: lat/long + 6 distances |
| Distances | Haversine, Earth radius 6,371 km, from the **locality centre** (`locality_centre_lat/long` of `add_fixed_features()`) to the six fixed landmarks of section 3; NaN when the centre is missing |
| Preprocessing (per fold) | numeric (incl. distances): median imputation → `StandardScaler`; binary passthrough; categorical one-hot; locality one-hot with `min_frequency=10` |
| Not used | listing pins, locality target encoding, any price-derived statistic, the test set |
| Metrics | MAE, RMSE, R² (rupees), Median APE; mean and sample std over the 5 folds |
| Saved outputs | none |

In [24]:
# Source files unchanged, byte for byte
for path, md5 in md5_at_start.items():
    assert file_md5(path) == md5, f"{path.name} changed!"
    print(f"{path.name:28} unchanged  ({md5})")

cleaned_listings.csv         unchanged  (3a81a385a9aca28d5b70322e1e143c08)
train_test_split.csv         unchanged  (10693e86d1d8a55e606399d718a8f556)
localities.csv               unchanged  (8d9bb54dfc2fb49673e74b976ab73ace)
build_features.py            unchanged  (ec9ba82a80e12d99798e512f510bfa84)


## Final comparison

In [25]:
final_table = pd.DataFrame([summarise(name, fold_tables[name]) for name in MODELS])
final_table["MAE"] = final_table["MAE (₹ lakh)"].round(3).astype(str) + " ± " + final_table["MAE std"].round(3).astype(str)
final_table["Median APE"] = (final_table["Median APE (%)"].round(3).astype(str) + " ± "
                             + final_table["Median APE std"].round(3).astype(str))
final_table["RMSE"] = final_table["RMSE (₹ lakh)"].round(3).astype(str) + " ± " + final_table["RMSE std"].round(3).astype(str)
final_table["R2"] = final_table["R²"].round(3).astype(str) + " ± " + final_table["R² std"].round(3).astype(str)
print("Mean ± std over the same 5 GroupKFold folds (training set only); MAE and RMSE in ₹ lakh, Median APE in %")
print(final_table[["Model", "MAE", "Median APE", "RMSE", "R2"]].to_string(index=False))

Mean ± std over the same 5 GroupKFold folds (training set only); MAE and RMSE in ₹ lakh, Median APE in %
                           Model            MAE     Median APE             RMSE            R2
          A. Baseline (lat/long) 70.248 ± 3.105 21.841 ± 0.204 179.528 ± 20.846 0.481 ± 0.028
               B. No coordinates  70.41 ± 3.054  22.007 ± 0.37  179.948 ± 20.93 0.478 ± 0.027
C. Distances instead of lat/long 69.365 ± 3.199  21.372 ± 0.43 177.689 ± 20.734 0.491 ± 0.029
         D. Lat/long + distances   69.32 ± 3.21 21.385 ± 0.412 177.667 ± 20.729 0.491 ± 0.029
